In [1]:
import pandas as pd
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
from tqdm import tqdm

In [2]:
df = pd.read_csv('../data/processed/filtered_complaints.csv')
print(df.shape)
df['product_category'].value_counts()

(335412, 9)


product_category
Savings Account    140319
Money Transfer      97188
Credit Card         80667
Personal Loan       17238
Name: count, dtype: int64

In [3]:
SAMPLE_SIZE = 12000

def stratified_sample(df, total_n, strat_col):
    fractions = df[strat_col].value_counts(normalize=True)
    samples = []
    for category, frac in fractions.items():
        n = int(round(frac * total_n))
        cat_df = df[df[strat_col] == category]
        samples.append(cat_df.sample(n=min(n, len(cat_df)), random_state=42))
    return pd.concat(samples).reset_index(drop=True)

sample_df = stratified_sample(df, SAMPLE_SIZE, 'product_category')
print(sample_df.shape)
print(sample_df['product_category'].value_counts())

(12000, 9)
product_category
Savings Account    5020
Money Transfer     3477
Credit Card        2886
Personal Loan       617
Name: count, dtype: int64


In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = []
metadatas = []

for idx, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    text_chunks = splitter.split_text(str(row['cleaned_narrative']))
    for i, chunk in enumerate(text_chunks):
        chunks.append(chunk)
        metadatas.append({
            "complaint_id": str(row['Complaint ID']),
            "product_category": row['product_category'],
            "issue": str(row.get('Issue', '')),
            "company": str(row.get('Company', '')),
            "chunk_index": i,
            "total_chunks": len(text_chunks)
        })

print(f"Total chunks created: {len(chunks)}")

100%|██████████| 12000/12000 [00:03<00:00, 3118.42it/s]

Total chunks created: 34844


In [5]:
from dotenv import load_dotenv
import os

load_dotenv()  

from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded. Embedding dimension:", model.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded. Embedding dimension: 384


C:\Users\HP EliteBook\AppData\Local\Temp\ipykernel_12156\566711332.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Model loaded. Embedding dimension:", model.get_sentence_embedding_dimension())


In [6]:
embeddings = model.encode(
    chunks,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)
print(embeddings.shape)

Batches:   0%|          | 0/545 [00:00<?, ?it/s]

(34844, 384)


In [7]:
client = chromadb.PersistentClient(path="../vector_store")

collection = client.get_or_create_collection(name="complaint_chunks")

ids = [f"chunk_{i}" for i in range(len(chunks))]

batch_size = 500
for i in tqdm(range(0, len(chunks), batch_size)):
    collection.add(
        ids=ids[i:i+batch_size],
        documents=chunks[i:i+batch_size],
        embeddings=embeddings[i:i+batch_size].tolist(),
        metadatas=metadatas[i:i+batch_size]
    )

print(f"Collection count: {collection.count()}")

100%|██████████| 70/70 [01:59<00:00,  1.71s/it]

Collection count: 34844


In [8]:
test_query = "unauthorized credit card charges"
test_embedding = model.encode([test_query]).tolist()

results = collection.query(
    query_embeddings=test_embedding,
    n_results=5
)

for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
    print(meta['product_category'], '|', meta['complaint_id'])
    print(doc[:150], '\n')

Savings Account | 9222991
. the following charges were not authorized by me and were made when my card was stolen 50.00 28.00 t 100.00 320.00 200.00 28.00 210.00 550.00 580.00 

Credit Card | 12325095
i had 2 unauthorized transactions on my card, for and for the charge they took care of no problem, but the charge they keep telling me it was authoriz 

Credit Card | 9943485
. i told him i never authorized this charge and kept the card in my possession all the time. he said not to worry and they would issue a new card repl 

Savings Account | 4586495
. when i called, i was told not to worry, they would send me a new card, and investigate the charges, and that i'd be fully credited for the fraudulen 

Credit Card | 9943485
. my immediate problems are my payment due date is close and i don't want deliquent charges. therefore, i even asked them at least to give a temporary 

